# Efficient Deep Learning for Brain Tumor MRI

**Goal:** teach a computer to look at a brain MRI scan and say which of four things it shows:
glioma tumor, meningioma tumor, pituitary tumor, or no tumor.

We build two models and compare them:
1. A standard **CNN** (convolutional neural network), the classic way to classify images.
2. A **DCFormer-style** network that uses *decomposed 1D depthwise convolutions*, which do a
   similar job with far fewer calculations (FLOPs).

Along the way we:
- find **duplicate images** that leak from the training set into the test set, and build a
  clean split so our accuracy numbers are honest,
- measure accuracy and AUC for both models,
- use **Grad-CAM** to see *where* each model looks in the scan,
- show how decomposition cuts compute for **3D** scans too.

Every cell has comments explaining each line. Run the cells top to bottom.

## 1. Setup

First we load the libraries we need and make the run **reproducible** (same random numbers
every time, so results can be repeated).

In [ ]:
# ---- Standard Python tools ----
import os          # work with files and folders
import random      # random numbers (used for shuffling)
import json        # save results as a .json text file
import itertools   # small helpers for loops (used when pairing duplicates)
from collections import Counter, defaultdict   # easy counting and grouping

# ---- Science / data tools ----
import numpy as np                  # arrays (grids of numbers)
import matplotlib.pyplot as plt     # drawing plots and showing images
from PIL import Image               # opening image files (jpg, png)

# ---- Deep learning (PyTorch) ----
import torch                        # the core library: tensors + GPU math
import torch.nn as nn               # "neural network" building blocks (layers)
import torch.nn.functional as F     # functions like relu, softmax
from torch.utils.data import Dataset, DataLoader   # feeding data to the model in batches

# ---- Metrics ----
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix

# ---- Reproducibility: fix every source of randomness to the same seed ----
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# ---- Pick the device: GPU if Colab gave us one, otherwise CPU ----
# A GPU does thousands of multiplications at once, so training is 10-50x faster on it.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

## 2. Download the dataset

The data comes from Kaggle (uploader: sartajbhuvaji). `kagglehub` downloads it and gives
us the folder path. You will be asked for your Kaggle username and API key the first time.

In [ ]:
# If the data is already on disk (set by the DATA_DIR environment variable), reuse it.
# Otherwise download from Kaggle. This lets the same notebook run anywhere.
if os.environ.get("DATA_DIR"):
    path = os.environ["DATA_DIR"]
else:
    import kagglehub
    kagglehub.login()
    path = kagglehub.dataset_download("sartajbhuvaji/brain-tumor-classification-mri")

print("Dataset folder:", path)

# os.path.join glues folder names into one address, adding the "/" for us.
train_dir = os.path.join(path, "Training")
test_dir  = os.path.join(path, "Testing")

# os.listdir returns a list of the names inside a folder.
print("Top folder holds:", os.listdir(path))
print("Training holds:  ", os.listdir(train_dir))

## 3. Explore the data

Inside `Training` and `Testing` there are four folders, one per class. **The folder name is
the label** (the correct answer) for every image inside it.

We walk every folder, collect each image's path and label, and count how many we have.

In [ ]:
# The four class names, in a fixed order. The position in this list becomes the class number:
# glioma_tumor = 0, meningioma_tumor = 1, no_tumor = 2, pituitary_tumor = 3.
# Models work with numbers, not words, so each class needs a number.
CLASSES = ["glioma_tumor", "meningioma_tumor", "no_tumor", "pituitary_tumor"]

def collect_images(root_dir, split_name):
    """
    Walk one top folder (Training or Testing) and return a list of records.
    Each record is a small dictionary: {"path": ..., "label": ..., "split": ...}.
    """
    records = []                                  # start with an empty list
    for label_idx, class_name in enumerate(CLASSES):    # label_idx = 0,1,2,3; class_name = folder name
        class_dir = os.path.join(root_dir, class_name)  # e.g. .../Training/glioma_tumor
        for fname in sorted(os.listdir(class_dir)):     # every file in that folder, in a fixed order
            records.append({
                "path": os.path.join(class_dir, fname), # full address of the image file
                "label": label_idx,                     # its class number
                "split": split_name,                    # "train" or "test" (the official split)
            })
    return records

train_records = collect_images(train_dir, "train")
test_records  = collect_images(test_dir,  "test")
all_records   = train_records + test_records           # one big list of every image

print("Official training images:", len(train_records))
print("Official test images:    ", len(test_records))
print("Total:                   ", len(all_records))

# Count images per class in each split. Counter counts how often each value appears.
for split_name, recs in [("train", train_records), ("test", test_records)]:
    counts = Counter(r["label"] for r in recs)
    print(f"\n{split_name} per class:")
    for idx, name in enumerate(CLASSES):
        print(f"   {name:18s} {counts[idx]}")

### Look at a few scans

Always look at your data before modelling. Here is one image from each class.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(14, 4))       # one row of 4 picture slots
for idx, class_name in enumerate(CLASSES):
    # Find the first training record whose label is this class.
    rec = next(r for r in train_records if r["label"] == idx)
    img = Image.open(rec["path"])                      # open the image file
    axes[idx].imshow(img, cmap="gray")                 # draw it (MRI scans are grayscale)
    axes[idx].set_title(f"{class_name}\n{img.size[0]}x{img.size[1]} px")  # size in pixels
    axes[idx].axis("off")                              # hide the axis ticks
plt.tight_layout()
plt.show()

# Image sizes vary across the dataset, so we will resize everything to one size later.

## 4. Find duplicate images (data leakage check)

**Leakage** = the model sees the answer during training. If a test image is an exact copy of
a training image, the model can "remember" it and the test accuracy is inflated.

To find copies we give every image a **fingerprint** called a *difference hash* (dHash):
shrink it to 9x8 grayscale, then for each of the 64 neighbouring pixel pairs write 1 if the
left pixel is brighter than the right one, else 0. Those 64 bits are the fingerprint.
Re-saving an image at another size or JPEG quality barely changes which pixel is brighter,
so true copies get the same fingerprint, while different scans almost never do.

In [ ]:
def image_fingerprint(img_path, hash_size=8):
    """Return a 64-bit difference hash that is the same for visually identical images."""
    img = Image.open(img_path).convert("L")                   # "L" = grayscale, one number per pixel
    img = img.resize((hash_size + 1, hash_size), Image.LANCZOS)   # 9 wide x 8 tall
    px = np.asarray(img, dtype=np.int16)                       # (8, 9) grid of brightness values
    diff = px[:, 1:] > px[:, :-1]                              # (8, 8) grid of True/False: brighter than left?
    bits = diff.flatten()                                      # 64 True/False values in a row
    return "".join("1" if b else "0" for b in bits)            # as a string of 0s and 1s

# Compute a fingerprint for every image and store it in its record.
for rec in all_records:
    rec["fp"] = image_fingerprint(rec["path"])

# Group records by fingerprint: dict of fingerprint -> list of records with that fingerprint.
groups = defaultdict(list)
for rec in all_records:
    groups[rec["fp"]].append(rec)

# Give each group a number ("group id"). Images in the same group are duplicates of each other.
for gid, (fp, recs) in enumerate(groups.items()):
    for rec in recs:
        rec["group"] = gid

n_unique = len(groups)
n_dup_images = sum(len(r) for r in groups.values() if len(r) > 1)
print("Unique images (groups):", n_unique)
print("Images that belong to a duplicate group:", n_dup_images)

# The key question: how many OFFICIAL TEST images also appear in the OFFICIAL TRAINING set?
train_fps = {r["fp"] for r in train_records}           # a set of all training fingerprints
leaked = [r for r in test_records if r["fp"] in train_fps]
leak_pct = 100 * len(leaked) / len(test_records)
print(f"\nTest images that duplicate a training image: {len(leaked)} / {len(test_records)} "
      f"= {leak_pct:.1f}%")

# Do duplicates ever disagree on the label? (Same scan filed under two classes = label noise.)
conflicts = sum(1 for recs in groups.values() if len({r['label'] for r in recs}) > 1)
print("Duplicate groups whose copies have different labels:", conflicts)

### Show one leaked pair

Proof that the leak is real: a test image and its copy in the training set.

In [ ]:
if leaked:
    test_rec = leaked[0]
    train_rec = next(r for r in train_records if r["fp"] == test_rec["fp"])
    fig, axes = plt.subplots(1, 2, figsize=(8, 4))
    axes[0].imshow(Image.open(train_rec["path"]), cmap="gray"); axes[0].set_title("Training copy")
    axes[1].imshow(Image.open(test_rec["path"]),  cmap="gray"); axes[1].set_title("Test copy")
    for ax in axes: ax.axis("off")
    plt.show()
    print("Train file:", os.path.basename(train_rec["path"]), "| Test file:", os.path.basename(test_rec["path"]))

## 5. Build a leakage-free split

We throw away the official train/test split and make our own, with one rule:
**all copies of the same image go to the same side.** We split by *group*, not by image.

- 70% of groups -> train (the model learns from these)
- 15% -> validation (we check progress and pick the best epoch)
- 15% -> test (touched only once, at the very end, for the final numbers)

We also keep the class proportions similar in each part (**stratified**).

In [ ]:
def grouped_stratified_split(records, train_frac=0.70, val_frac=0.15, seed=SEED):
    """Split by duplicate group so no image (or its copy) appears on two sides."""
    rng = random.Random(seed)

    # One representative label per group (use the majority label if copies disagree).
    group_label = {}
    for gid in {r["group"] for r in records}:
        labels = [r["label"] for r in records if r["group"] == gid]
        group_label[gid] = Counter(labels).most_common(1)[0][0]

    split_of_group = {}
    # Do the split separately inside each class so every class is spread across train/val/test.
    for cls in range(len(CLASSES)):
        gids = [g for g, l in group_label.items() if l == cls]
        rng.shuffle(gids)                                   # random order
        n = len(gids)
        n_train = int(n * train_frac)
        n_val   = int(n * val_frac)
        for i, g in enumerate(gids):
            if i < n_train:            split_of_group[g] = "train"
            elif i < n_train + n_val:  split_of_group[g] = "val"
            else:                      split_of_group[g] = "test"

    # Every record inherits the split of its group.
    for rec in records:
        rec["my_split"] = split_of_group[rec["group"]]
    return records

all_records = grouped_stratified_split(all_records)

# Keep exactly ONE copy per group so duplicates don't give extra weight to the same scan.
seen = set()
clean_records = []
for rec in all_records:
    if rec["group"] not in seen:
        seen.add(rec["group"])
        clean_records.append(rec)

splits = {name: [r for r in clean_records if r["my_split"] == name] for name in ["train", "val", "test"]}
for name, recs in splits.items():
    print(f"{name:5s}: {len(recs):4d} images  |  per class:",
          [sum(1 for r in recs if r['label'] == c) for c in range(len(CLASSES))])

# Sanity check: no fingerprint should appear in two different splits.
fps_by_split = {name: {r["fp"] for r in recs} for name, recs in splits.items()}
assert not (fps_by_split["train"] & fps_by_split["test"]), "LEAK: train and test share an image!"
assert not (fps_by_split["train"] & fps_by_split["val"]),  "LEAK: train and val share an image!"
print("\nLeakage check passed: no image appears in two splits.")

## 6. Feed images to PyTorch

PyTorch needs numbers, not jpg files. Three ideas:

- A **tensor** is PyTorch's version of a NumPy array. It can live on the GPU.
- A **Dataset** is a Python class that knows how many examples there are (`__len__`) and how
  to load example number *i* (`__getitem__`).
- A **DataLoader** pulls examples from the Dataset in **batches** (groups, here 32 at a time)
  and shuffles them.

Each image becomes a tensor of shape `(1, 128, 128)`: 1 channel (grayscale), 128 rows,
128 columns. We scale pixel values from 0-255 down to 0-1 and then **normalize** so the
average is about 0 and the spread about 1. Networks train faster on numbers in that range.

In [ ]:
IMG_SIZE = 128     # every image is resized to 128x128 pixels (small = fast training)
BATCH_SIZE = 32    # the model looks at 32 images at a time

class MRIDataset(Dataset):
    """Turns our list of records into something a DataLoader can read."""

    def __init__(self, records, augment=False):
        self.records = records
        self.augment = augment        # random flips during training only

    def __len__(self):
        return len(self.records)      # how many images

    def __getitem__(self, i):
        rec = self.records[i]
        img = Image.open(rec["path"]).convert("L").resize((IMG_SIZE, IMG_SIZE))   # grayscale, resized
        x = torch.from_numpy(np.asarray(img, dtype=np.float32) / 255.0)          # 0..255 -> 0.0..1.0
        x = (x - 0.5) / 0.5                     # normalize to roughly -1..1
        x = x.unsqueeze(0)                      # add the channel dimension: (128,128) -> (1,128,128)
        if self.augment and random.random() < 0.5:
            x = torch.flip(x, dims=[2])         # flip left-right half the time (more variety for free)
        y = rec["label"]                        # the class number
        return x, y

train_ds = MRIDataset(splits["train"], augment=True)
val_ds   = MRIDataset(splits["val"])
test_ds  = MRIDataset(splits["test"])

# shuffle=True for training so each epoch sees the images in a new order.
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

# Peek at one batch to confirm the shapes.
xb, yb = next(iter(train_loader))
print("Batch of images:", xb.shape)   # (32, 1, 128, 128) = 32 images, 1 channel, 128x128
print("Batch of labels:", yb.shape)   # (32,)             = 32 class numbers
print("First 8 labels:", yb[:8].tolist())

## 7. Model 1: a standard CNN

A **convolution** slides a small window (here 3x3) over the image and computes a weighted
sum at every position. The weights are learned. Each window learns to detect one pattern
(an edge, a bright blob...). Stacking convolutions lets later layers detect bigger patterns
built from the small ones.

Our baseline is four blocks of: `Conv -> BatchNorm -> ReLU -> MaxPool`.
- **BatchNorm** keeps the numbers well-scaled so training is stable.
- **ReLU** turns negatives to 0 (adds non-linearity so the net can learn curves, not just lines).
- **MaxPool** halves the width and height (keeps the strongest signal in each 2x2 square).

At the end, **global average pooling** squashes each channel map to one number, and a
**Linear** layer turns those numbers into 4 scores, one per class.

In [ ]:
class BaselineCNN(nn.Module):
    def __init__(self, num_classes=4):
        super().__init__()                     # set up the nn.Module machinery
        chans = [1, 32, 64, 128, 256]          # channels in -> out for the four blocks
        blocks = []
        for cin, cout in zip(chans[:-1], chans[1:]):
            blocks += [
                nn.Conv2d(cin, cout, kernel_size=3, padding=1),  # 3x3 window; padding keeps size
                nn.BatchNorm2d(cout),
                nn.ReLU(inplace=True),
                nn.MaxPool2d(2),                                  # 128 -> 64 -> 32 -> 16 -> 8
            ]
        self.features = nn.Sequential(*blocks)          # run the blocks one after another
        self.pool = nn.AdaptiveAvgPool2d(1)             # (B,256,8,8) -> (B,256,1,1)
        self.classifier = nn.Linear(chans[-1], num_classes)   # 256 numbers -> 4 class scores

    def forward(self, x):                               # how data flows through the model
        x = self.features(x)
        x = self.pool(x).flatten(1)                     # (B,256,1,1) -> (B,256)
        return self.classifier(x)                       # (B,4) raw scores ("logits")

cnn = BaselineCNN().to(device)
print(cnn)
print("Parameters:", sum(p.numel() for p in cnn.parameters()))

## 8. Model 2: DCFormer-style network (decomposed 1D depthwise convolutions)

A normal 3x3 convolution with `C` input and `C` output channels costs about `9 * C * C`
multiplications per pixel. Two tricks make it far cheaper:

1. **Depthwise**: each channel gets its *own* small filter, no mixing between channels.
   Cost drops to `9 * C` per pixel. A cheap 1x1 ("pointwise") convolution then mixes channels.
2. **Decomposed**: replace the 3x3 window with a 1x3 (horizontal) followed by a 3x1 (vertical).
   Cost drops from 9 to 3 + 3 = 6 weights per channel per pixel.

Together: a **decomposed depthwise 1D convolution**. This is the idea behind DCFormer, which
applies it in 2D and 3D. Our block is:
`DW 1x3 -> DW 3x1 -> BatchNorm -> Pointwise 1x1 -> GELU -> Pointwise 1x1`, plus a **residual**
connection (add the input back) so gradients flow easily through many blocks.

In [ ]:
class DecomposedDWBlock(nn.Module):
    """One DCFormer-style block for 2D images."""
    def __init__(self, dim, k=7):
        super().__init__()
        # groups=dim makes the conv DEPTHWISE: each channel has its own filter.
        self.dw_h = nn.Conv2d(dim, dim, kernel_size=(1, k), padding=(0, k // 2), groups=dim)  # 1 x k
        self.dw_v = nn.Conv2d(dim, dim, kernel_size=(k, 1), padding=(k // 2, 0), groups=dim)  # k x 1
        self.norm = nn.BatchNorm2d(dim)
        self.pw1 = nn.Conv2d(dim, dim * 2, kernel_size=1)   # pointwise: mix channels, widen
        self.act = nn.GELU()                                 # smooth ReLU-like activation
        self.pw2 = nn.Conv2d(dim * 2, dim, kernel_size=1)   # pointwise: back to dim channels

    def forward(self, x):
        shortcut = x                     # remember the input
        x = self.dw_h(x)                 # horizontal 1D depthwise conv
        x = self.dw_v(x)                 # vertical 1D depthwise conv
        x = self.norm(x)
        x = self.pw2(self.act(self.pw1(x)))
        return x + shortcut              # residual: output = input + what the block learned

class DCFormerLite(nn.Module):
    """Stem -> 4 stages of decomposed blocks, each stage halves the image and widens channels."""
    def __init__(self, num_classes=4, dims=(24, 48, 96, 192), depth=1):
        super().__init__()
        self.stem = nn.Sequential(                         # 128x128x1 -> 32x32x24 quickly
            nn.Conv2d(1, dims[0], kernel_size=4, stride=4),
            nn.BatchNorm2d(dims[0]),
        )
        stages = []
        for i, d in enumerate(dims):
            if i > 0:                                      # downsample between stages
                stages.append(nn.Sequential(
                    nn.BatchNorm2d(dims[i - 1]),
                    nn.Conv2d(dims[i - 1], d, kernel_size=2, stride=2)))  # halve size, widen
            stages += [DecomposedDWBlock(d) for _ in range(depth)]
        self.stages = nn.Sequential(*stages)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Linear(dims[-1], num_classes)

    def forward(self, x):
        x = self.stages(self.stem(x))
        return self.classifier(self.pool(x).flatten(1))

dcf = DCFormerLite().to(device)
print("DCFormer-lite parameters:", sum(p.numel() for p in dcf.parameters()))

## 9. Count FLOPs

**FLOPs** = floating-point operations, a device-independent measure of how much computing a
model needs for one image. We attach a small hook to every Conv and Linear layer that
records `output_elements * (kernel_elements * in_channels / groups)` multiply-adds.

In [ ]:
def count_flops(model, input_shape):
    """Return multiply-add count for one forward pass with an input of the given shape."""
    flops = []

    def conv_hook(module, inp, out):
        k = int(np.prod(module.kernel_size))                 # 9 for 3x3, 3 for 1x3, 27 for 3x3x3
        cin_per_group = module.in_channels // module.groups  # 1 for depthwise, C for normal
        flops.append(out.numel() * k * cin_per_group)        # one MAC per kernel weight per output

    def linear_hook(module, inp, out):
        flops.append(module.in_features * module.out_features)

    handles = []
    for m in model.modules():
        if isinstance(m, (nn.Conv2d, nn.Conv3d)):
            handles.append(m.register_forward_hook(conv_hook))
        elif isinstance(m, nn.Linear):
            handles.append(m.register_forward_hook(linear_hook))

    model.eval()
    with torch.no_grad():                                   # no gradients needed, just a pass
        model(torch.zeros(1, *input_shape, device=next(model.parameters()).device))
    for h in handles:
        h.remove()                                          # clean up the hooks
    return sum(flops)

cnn_flops = count_flops(cnn, (1, IMG_SIZE, IMG_SIZE))
dcf_flops = count_flops(dcf, (1, IMG_SIZE, IMG_SIZE))
print(f"Baseline CNN:   {cnn_flops/1e6:8.1f} MFLOPs")
print(f"DCFormer-lite:  {dcf_flops/1e6:8.1f} MFLOPs")
print(f"CNN needs {cnn_flops/dcf_flops:.1f}x more compute than DCFormer-lite")

## 10. Training

One **epoch** = one pass over all training images. For each batch:
1. **Forward**: the model makes predictions.
2. **Loss**: cross-entropy measures how wrong the predictions are (0 = perfect).
3. **Backward**: autograd computes, for every weight, which direction reduces the loss.
4. **Step**: the optimizer (AdamW) nudges every weight a little in that direction.

After each epoch we check **validation** accuracy. We keep the weights from the best epoch,
which protects against **overfitting** (memorizing the training images instead of learning).

In [ ]:
EPOCHS = int(os.environ.get("EPOCHS", 15))       # 15 is a good balance on a free Colab GPU

def run_epoch(model, loader, optimizer=None):
    """Run one pass over `loader`. If optimizer is given, train; otherwise just evaluate."""
    training = optimizer is not None
    model.train(training)                         # train mode: BatchNorm/augment behave differently
    total_loss, all_preds, all_labels = 0.0, [], []
    with torch.set_grad_enabled(training):        # gradients only while training
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device) # move the batch onto the GPU
            logits = model(xb)                    # forward pass: (B,4) scores
            loss = F.cross_entropy(logits, yb)    # how wrong are we on this batch?
            if training:
                optimizer.zero_grad()             # clear old gradients
                loss.backward()                   # compute new gradients
                optimizer.step()                  # update the weights
            total_loss += loss.item() * len(xb)
            all_preds.append(logits.argmax(1).cpu())   # predicted class = index of highest score
            all_labels.append(yb.cpu())
    preds, labels = torch.cat(all_preds), torch.cat(all_labels)
    acc = (preds == labels).float().mean().item()
    return total_loss / len(loader.dataset), acc

def train_model(model, name):
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    # Cosine schedule: learning rate starts at 1e-3 and glides down to ~0 by the last epoch.
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
    best_acc, best_state, history = 0.0, None, []
    for epoch in range(1, EPOCHS + 1):
        tr_loss, tr_acc = run_epoch(model, train_loader, optimizer)
        va_loss, va_acc = run_epoch(model, val_loader)
        scheduler.step()
        history.append({"epoch": epoch, "train_loss": tr_loss, "train_acc": tr_acc,
                        "val_loss": va_loss, "val_acc": va_acc})
        if va_acc > best_acc:                          # remember the best weights so far
            best_acc = va_acc
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        print(f"[{name}] epoch {epoch:2d} | train loss {tr_loss:.3f} acc {tr_acc:.3f} "
              f"| val loss {va_loss:.3f} acc {va_acc:.3f}")
    model.load_state_dict(best_state)                  # restore the best epoch
    torch.save(best_state, f"{name}_best.pt")          # save to disk too
    print(f"[{name}] best val acc: {best_acc:.3f}")
    return history

cnn_history = train_model(cnn, "cnn")

In [ ]:
dcf_history = train_model(dcf, "dcformer")

### Training curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for hist, label in [(cnn_history, "CNN"), (dcf_history, "DCFormer-lite")]:
    axes[0].plot([h["val_loss"] for h in hist], label=label)
    axes[1].plot([h["val_acc"]  for h in hist], label=label)
axes[0].set_title("Validation loss (lower is better)"); axes[0].set_xlabel("epoch"); axes[0].legend()
axes[1].set_title("Validation accuracy");                axes[1].set_xlabel("epoch"); axes[1].legend()
plt.tight_layout(); plt.show()

## 11. Final evaluation on the clean test set

- **Accuracy**: fraction of test images with the correct predicted class.
- **AUC** (area under the ROC curve): how well the model *ranks* images by class probability.
  1.0 is perfect, 0.5 is random. For 4 classes we average the one-vs-rest AUCs.
- **Confusion matrix**: rows = true class, columns = predicted class. The diagonal is correct.

In [ ]:
def evaluate(model, loader):
    model.eval()
    probs, labels = [], []
    with torch.no_grad():
        for xb, yb in loader:
            logits = model(xb.to(device))
            probs.append(F.softmax(logits, dim=1).cpu())   # softmax: scores -> probabilities summing to 1
            labels.append(yb)
    probs, labels = torch.cat(probs).numpy(), torch.cat(labels).numpy()
    preds = probs.argmax(1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "auc": roc_auc_score(labels, probs, multi_class="ovr"),
        "confusion": confusion_matrix(labels, preds, labels=list(range(len(CLASSES)))),
    }

results = {}
for model, name, flops in [(cnn, "Baseline CNN", cnn_flops), (dcf, "DCFormer-lite", dcf_flops)]:
    r = evaluate(model, test_loader)
    r["mflops"] = flops / 1e6
    r["params"] = sum(p.numel() for p in model.parameters())
    results[name] = r
    print(f"{name:14s} | accuracy {r['accuracy']*100:5.1f}% | AUC {r['auc']:.3f} "
          f"| {r['mflops']:7.1f} MFLOPs | {r['params']/1e6:.2f}M params")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, (name, r) in zip(axes, results.items()):
    cm = r["confusion"]
    ax.imshow(cm, cmap="Blues")
    for i in range(len(CLASSES)):
        for j in range(len(CLASSES)):
            ax.text(j, i, cm[i, j], ha="center", va="center",
                    color="white" if cm[i, j] > cm.max() / 2 else "black")
    ax.set_xticks(range(4)); ax.set_xticklabels([c.replace("_tumor", "") for c in CLASSES], rotation=30)
    ax.set_yticks(range(4)); ax.set_yticklabels([c.replace("_tumor", "") for c in CLASSES])
    ax.set_xlabel("predicted"); ax.set_ylabel("true"); ax.set_title(name)
plt.tight_layout(); plt.show()

## 12. Grad-CAM: where does each model look?

**Grad-CAM** highlights the regions of the input that most increased the score for the
predicted class. Steps:
1. Run the image through the model, keeping the feature maps `A` of the last conv stage.
2. Compute the gradient of the predicted-class score with respect to `A`.
3. Average that gradient over space to get one weight per channel.
4. Weighted-sum the channels, keep positives (ReLU), resize to the image, overlay as heat.

If the hot region sits on the tumor, the model is using the right evidence.

In [ ]:
class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.activations = None
        self.gradients = None
        # Hooks let us grab what flows through target_layer during forward and backward passes.
        target_layer.register_forward_hook(self._save_act)
        target_layer.register_full_backward_hook(self._save_grad)

    def _save_act(self, module, inp, out):   self.activations = out.detach()
    def _save_grad(self, module, gin, gout): self.gradients = gout[0].detach()

    def __call__(self, x):
        self.model.eval()
        x = x.to(device).unsqueeze(0)                     # (1,1,H,W): a batch of one
        logits = self.model(x)
        cls = logits.argmax(1).item()                     # predicted class
        self.model.zero_grad()
        logits[0, cls].backward()                         # gradient of that class score
        weights = self.gradients.mean(dim=(2, 3), keepdim=True)    # one weight per channel
        cam = (weights * self.activations).sum(1, keepdim=True)    # weighted sum of channels
        cam = F.relu(cam)                                           # keep only positive evidence
        cam = F.interpolate(cam, size=(IMG_SIZE, IMG_SIZE), mode="bilinear", align_corners=False)
        cam = cam[0, 0].cpu().numpy()
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)   # scale to 0..1
        return cam, cls

# Last conv stage of each model: the final feature maps before pooling.
cam_cnn = GradCAM(cnn, cnn.features[-4])          # the last Conv2d in the baseline
cam_dcf = GradCAM(dcf, dcf.stages[-1])            # the last decomposed block

n_show = 4
fig, axes = plt.subplots(n_show, 3, figsize=(10, 3.2 * n_show))
for row in range(n_show):
    x, y = test_ds[row * (len(test_ds) // n_show)]   # spread picks across the test set
    img = x[0].numpy()
    axes[row, 0].imshow(img, cmap="gray"); axes[row, 0].set_title(f"true: {CLASSES[y]}")
    for col, (cam_obj, name) in enumerate([(cam_cnn, "CNN"), (cam_dcf, "DCFormer-lite")], start=1):
        cam, pred = cam_obj(x)
        axes[row, col].imshow(img, cmap="gray")
        axes[row, col].imshow(cam, cmap="jet", alpha=0.45)       # heat overlay
        axes[row, col].set_title(f"{name} -> {CLASSES[pred]}")
    for ax in axes[row]: ax.axis("off")
plt.tight_layout(); plt.show()

## 13. 3D study: how much compute does decomposition save on volumes?

Real MRI scans are 3D volumes (many slices). A full 3x3x3 convolution costs 27 weights per
channel per voxel. Decomposing into three 1D depthwise convs (3x1x1, 1x3x1, 1x1x3) costs
3 + 3 + 3 = 9, and depthwise removes the channel-mixing factor too. We measure it with the
same FLOP counter on a dummy 64x64x64 volume.

In [ ]:
class Full3DBlock(nn.Module):
    """Standard 3D conv: mixes all channels with a 3x3x3 window."""
    def __init__(self, c):
        super().__init__()
        self.conv = nn.Conv3d(c, c, kernel_size=3, padding=1)
    def forward(self, x): return self.conv(x)

class Decomposed3DBlock(nn.Module):
    """Three depthwise 1D convs (one per axis) + a pointwise 1x1x1 conv to mix channels."""
    def __init__(self, c):
        super().__init__()
        self.d = nn.Conv3d(c, c, kernel_size=(3, 1, 1), padding=(1, 0, 0), groups=c)
        self.h = nn.Conv3d(c, c, kernel_size=(1, 3, 1), padding=(0, 1, 0), groups=c)
        self.w = nn.Conv3d(c, c, kernel_size=(1, 1, 3), padding=(0, 0, 1), groups=c)
        self.pw = nn.Conv3d(c, c, kernel_size=1)
    def forward(self, x): return self.pw(self.w(self.h(self.d(x))))

C, D = 32, 64                                          # 32 channels, 64x64x64 volume
full_flops = count_flops(Full3DBlock(C).to(device),       (C, D, D, D))
deco_flops = count_flops(Decomposed3DBlock(C).to(device), (C, D, D, D))
saving = 100 * (1 - deco_flops / full_flops)
print(f"Full 3x3x3 conv block:        {full_flops/1e9:6.2f} GFLOPs")
print(f"Decomposed 1D depthwise block: {deco_flops/1e9:6.2f} GFLOPs")
print(f"Decomposition cuts 3D compute by {saving:.0f}%")

## 14. Save results

Everything a reader needs, in one small JSON file for the README.

In [ ]:
summary = {
    "dataset": {
        "official_train": len(train_records),
        "official_test": len(test_records),
        "unique_images": n_unique,
        "test_images_duplicated_in_train_pct": round(leak_pct, 1),
        "clean_split": {k: len(v) for k, v in splits.items()},
    },
    "models": {name: {"accuracy_pct": round(r["accuracy"] * 100, 1),
                      "auc": round(r["auc"], 3),
                      "mflops": round(r["mflops"], 1),
                      "params": r["params"]} for name, r in results.items()},
    "flops_ratio_cnn_over_dcformer": round(cnn_flops / dcf_flops, 1),
    "3d_decomposition_compute_saving_pct": round(saving, 0),
}
with open("results.json", "w") as f:
    json.dump(summary, f, indent=2)
print(json.dumps(summary, indent=2))